# BBEH × PromptPotter

Runs PromptPotter's L1/L2/L3 optimization loop on BBEH using `gpt-oss-120b` via Groq, producing a `results_potter.json` next to `results_capo.json` / `results_dspy.json`.

**Methodology.** One global prompt is optimized on the full mini-BBEH pool (460 examples, pooled across all 23 tasks) and then evaluated on the held-out non-mini set (~4,060 examples, i.e. every BBEH example NOT in mini). The mini/non-mini partition is a HuggingFace-native flag on each record, so the train/test split is disjoint by construction — zero leakage. This matches how BBEH is officially graded: one model, one prompt, per-task accuracies reported for the harmonic-mean metric.

Not a per-task loop. Specialising a different prompt per task inflates the score relative to what you'd actually deploy, and with 20 mini examples per task the per-task optimizer hits noise-level 100% and early-stops on round 1 without learning anything.

**Runs locally against this repo** (unlike the Colab-based CAPO/DSPy notebooks). Prereqs:

- `pip install -e ".[dev,jupyter]"` from the repo root
- `datasets` package: `pip install datasets`
- `.env` with `GROQ_API_KEY`
- A running PromptPotter-compatible backend at `http://127.0.0.1:8000` exposing an `llm_only` node. BBEH's `datasets/bbeh/pipeline.json` constrains the active pipeline to that single node, so every query is a plain LLM call with the optimized prompt as system message.

**Hyperparameters** (`MAX_ROUNDS`, `N_VARIANTS`, `SP_BUDGET_TTEST`) are **unmeasured starting points**, not tuned values — this is a pre-hyperparameter-measurement run.

**Before running the full campaign, smoke-test first**: `python scripts/smoke_campaign.py --dataset bbeh` (~90s).

In [1]:
# Cell 1 — env + autoreload + path setup
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Notebook lives in `notebooks/`. Repo root is one level up. `shared_config.py`
# and the sibling `results_*.json` files stay in `docs/research/bbeh-comparison/`
# because the CAPO / DSPy comparison notebooks live there too.
_REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_BBEH_DIR = _REPO_ROOT / "docs" / "research" / "bbeh-comparison"
if str(_BBEH_DIR) not in sys.path:
    sys.path.insert(0, str(_BBEH_DIR))

try:
    from dotenv import load_dotenv
    load_dotenv(_REPO_ROOT / ".env")
except ImportError:
    pass

assert os.environ.get("GROQ_API_KEY"), "GROQ_API_KEY missing from environment"
print("env OK")

env OK


In [2]:
# Cell 2 — PromptPotter notebook API + BBEH data
from promptpotter.presentation.ui.campaign import (
    init_services,
    prepare_scoring_context,
    run_optimization_notebook,
    show_campaign_summary,
    configure_pipeline,
)
from promptpotter.shared.scoring import SCORING_FUNCTIONS

from shared_config import (
    MODEL_ID,
    SPLIT_SEED,
    load_and_split,
    export_results,
)

assert "exact_match" in SCORING_FUNCTIONS, "exact_match scorer missing from registry"
exact_match = SCORING_FUNCTIONS["exact_match"]

train_pool, test_by_task = load_and_split()
tasks = sorted(test_by_task.keys())

# Sanity: mini/non-mini partition must be disjoint.
_train_keys = {(ex["input"], ex["target"]) for ex in train_pool}
_test_keys = {
    (ex["input"], ex["target"])
    for items in test_by_task.values()
    for ex in items
}
assert not (_train_keys & _test_keys), "LEAK: train and test overlap"

n_test = sum(len(v) for v in test_by_task.values())
print(
    f"Loaded BBEH: {len(train_pool)} train (mini, pooled), "
    f"{n_test} test (non-mini, per-task) across {len(tasks)} tasks (seed={SPLIT_SEED})"
)

C:\Users\dsacc\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded BBEH: 460 train (mini, pooled), 4060 test (non-mini, per-task) across 23 tasks (seed=42)


In [3]:
# Cell 3 — campaign config (single global campaign, not per-task)
#
# These three knobs are the most expensive dials; they are *unmeasured* starting
# points, not tuned values. A later hyperparameter sweep will replace them.
MAX_ROUNDS = 8
N_VARIANTS = 4
SP_BUDGET_TTEST = 15  # rolling sample per round; Welch t-test early-stops losers

def build_campaign_config() -> dict:
    return {
        "dataset_name": "bbeh",
        "scoring": "exact_match(predicted, ground_truth)",
        "sp_budget_ttest": SP_BUDGET_TTEST,
        "recon_sample_size": SP_BUDGET_TTEST,
        "exclude_nodes": [],
        "pipeline_overrides": {},
        "task_context": {
            "task_description": (
                "Solve a reasoning problem from BIG-Bench Extra Hard (BBEH), which spans 23 "
                "diverse task types including boardgame QA, multi-step arithmetic, causal "
                "reasoning, disambiguation, and adversarial distractor text. Read the input "
                "carefully, reason step by step as needed, and return only the final answer."
            ),
        },
        "optimization": {
            "l1_patience": 2,
            "max_rounds": MAX_ROUNDS,
            "n_variants": N_VARIANTS,
            "creativity": 0.7,
            "improvement_threshold": 0.01,
            "seed": 42,
            "max_failures": 10,
            "degradation_threshold": 0.4,
            "enable_l2": True,
            "enable_l3": True,
            "l2_patience": 5,
            "l3_patience": 3,
            "l2_temperature": 0.3,
            "l3_temperature": 0.5,
            "enable_critique": True,
        },
        "optimizer_llm": {
            "model": "openai/gpt-oss-120b",
            "provider": "groq",
            "temperature": 0.4,
            "max_tokens": 2000,
        },
        "pipeline_params": None,
    }

def normalize(examples):
    """BBEH {input, target} -> PromptPotter {query, ground_truth}."""
    return [
        {"query": ex["input"], "ground_truth": ex["target"]}
        for ex in examples
    ]

print(
    f"Global campaign: max_rounds={MAX_ROUNDS}, n_variants={N_VARIANTS}, "
    f"sp_budget_ttest={SP_BUDGET_TTEST}, train={len(train_pool)}, test={n_test}"
)

Global campaign: max_rounds=8, n_variants=4, sp_budget_ttest=15, train=460, test=4060


In [5]:
# Cell 4 — run the campaign end-to-end
#
# One cell: prepare scoring context → run L1/L2/L3 optimization →
# per-task test evaluation → export results_potter.json.

from promptpotter.domain.opt_search_point import PromptTemplate

train_norm = normalize(train_pool)
test_norm_by_task = {task: normalize(items) for task, items in test_by_task.items()}

print(f"
{'=' * 60}
GLOBAL OPTIMIZATION ({len(train_norm)} train samples)
{'=' * 60}")

session = await init_services(
    backend_url="http://127.0.0.1:8000",
    dataset_name="bbeh",
)
campaign_config = build_campaign_config()
pipeline_params = configure_pipeline(session, campaign_config)

# -- Load baseline + auto-score 15-query anchor on the t-test slice --------
baseline_sp, dataset_obj, campaign_rounds, _ = await prepare_scoring_context(
    session,
    train_norm,
    campaign_config,
    run_baseline=False,
    pipeline_params=pipeline_params,
)
baseline_train_acc = campaign_rounds[0]["accuracy"] if campaign_rounds else 0.0
# -- Run the L1/L2/L3 optimization loop ------------------------------------
campaign_rounds, cycle_result = await run_optimization_notebook(
    campaign_rounds,
    dataset_obj,
    campaign_config,
    session=session,
    experiment_id="",
)

winner_prompt_fields = cycle_result.winner_prompt_fields
winner_pipeline_params = cycle_result.winner_pipeline_params
train_acc = cycle_result.best_accuracy

# -- Per-task test evaluation of the global winner -------------------------
print(f"\n{'=' * 60}\nPER-TASK TEST EVALUATION\n{'=' * 60}")

per_task_results: dict[str, dict] = {}
for i, task in enumerate(tasks, start=1):
    test_items = test_norm_by_task[task]
    hits = 0
    for ex in test_items:
        resp = await session.backend_client.run_query(
            ex["query"], pipeline_params=winner_pipeline_params
        )
        ranking = resp.get("data", {}).get("final_ranking") or []
        predicted = ranking[0].get("candidate", "") if ranking else ""
        hits += int(exact_match(predicted, ex["ground_truth"]))

    acc = hits / len(test_items) if test_items else 0.0
    per_task_results[task] = {"accuracy": round(acc, 4), "n_test": len(test_items)}
    print(f"  [{i:2d}/{len(tasks)}] {task:<40s} {acc:>6.1%}  ({hits}/{len(test_items)})")

await session.backend_client.aclose()

macro_avg = sum(r["accuracy"] for r in per_task_results.values()) / len(per_task_results)
total_test = sum(r["n_test"] for r in per_task_results.values())
print(
    f"\nMacro-avg test accuracy: {macro_avg:.1%}  "
    f"(global winner, {total_test} non-mini examples across {len(tasks)} tasks)"
)

# -- Export results_potter.json next to results_capo.json / results_dspy.json
winner_prompt_str = PromptTemplate(**winner_prompt_fields).render()
optimized_prompts = {"__global__": winner_prompt_str}

export_results(
    method="promptpotter",
    per_task=per_task_results,
    config={
        "optimizer": "promptpotter",
        "max_rounds": MAX_ROUNDS,
        "n_variants": N_VARIANTS,
        "sp_budget_ttest": SP_BUDGET_TTEST,
        "model_id": MODEL_ID,
        "n_train": len(train_pool),
        "train_accuracy": round(train_acc, 4),
        "baseline_train_accuracy": round(baseline_train_acc, 4),
        "rounds": len(campaign_rounds),
        "methodology": (
            "Single global prompt optimized on 460 mini-BBEH examples pooled "
            "across 23 tasks; evaluated on all non-mini examples (~4,060). "
            "Mini/non-mini partition is disjoint by HF flag — no leakage."
        ),
        "note": "unmeasured starting hyperparameters — pre-sweep",
    },
    optimized_prompts=optimized_prompts,
    output_path=str(_BBEH_DIR / "results_potter.json"),
)


2026-04-15 15:58:01 INFO     [promptpotter.application.pipeline_discovery] Parsed pipeline 'BBEH' with 1 steps
2026-04-15 15:58:01 INFO     [promptpotter.application.campaign.campaign_setup] Static pipeline schema loaded: bbeh vv0.1
2026-04-15 15:58:01 INFO     [promptpotter.application.campaign.campaign_setup] Loaded dataset 'bbeh' from store: 460 items, 213 session terms
2026-04-15 15:58:01 INFO     [promptpotter.application.campaign.campaign_setup] Baseline loaded from canonical store: datasets/bbeh/prompts/ → llm_only
2026-04-15 15:58:01 INFO     [promptpotter.application.campaign.data] Auto-baseline on t-test slice (15/460 queries) — L1 reference anchor



GLOBAL OPTIMIZATION (460 train samples, seed=fresh)
Pipeline: bbeh (1 nodes)
Backend: http://127.0.0.1:8000
Dataset: bbeh (460 queries)
Dataset    : bbeh (460 queries)
Session terms: 213
Starting prompt: bbeh/prompts/[llm_only|default].json → llm_only
Active nodes: llm_only


Baseline eval: 100%|██████████| 15/15 [00:00<00:00, 416.85query/s]


  0.0s MISS [ai]📖 -> 'C' gt:'(E)' q:'Here is a sentence with pronoun(s) whose'
  0.0s MISS [ai]📖 -> '453' gt:'581' q:'Consider the following new operations:  '
  0.0s MISS [ai]📖 -> '06-02-2006' gt:'05-30-2006' q:'Let the answer to Question1 be X.  Quest'
  0.0s MISS [ai]📖 -> 'K' gt:'(A)' q:'Suppose we draw this SVG path element: M'
  0.0s HIT [ai]📖 -> '5' gt:'5' q:'You are an expert in a language called d'
  0.0s HIT [ai]📖 -> '120, 1' gt:'120, 1' q:'Olivia, Amelia, Victoria, Karen, Rachel '
  0.0s MISS [ai]📖 -> 'unknown, unknown, yes' gt:'yes, yes, yes' q:'In this question, assume each person eit'
  0.0s MISS [ai]📖 -> 'No' gt:'Yes' q:'Question: David has a new dryer in his a'
  0.0s MISS [ai]📖 -> '20' gt:'18' q:'You are an expert in a language called d'
  0.0s HIT [ai]📖 -> '422.15' gt:'422.15' q:'I have a table with 27 rows (including t'
  0.0s MISS [ai]📖 -> '0' gt:'253' q:'Consider the following new operations:  '
  0.0s MISS [ai]📖 -> '113' gt:'259' q:'I have 92 huawei nova 11 se mobi

2026-04-15 15:58:02 WARNING  [promptpotter.infrastructure.tracing.sinks.langfuse_sink] Skipping Langfuse cloud dataset registration for 459 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-04-15 15:58:02 INFO     [promptpotter.infrastructure.store.dataset_run_store] Registered prompt alias: 6cc86028 ↔ 1218ac7c


...

2026-04-15 16:01:52 WARNING  [promptpotter.application.scoring.sample_measurement] measure_sample failed for You are an expert in a language called dyck where you must c: 


  [  2]       MISS  ERR:'unknown error' gt:'6' q:'You are an expert in a language called d'


2026-04-15 16:01:53 WARNING  [promptpotter.application.scoring.search_point_scorer] Query loop force-interrupted at query 2/15.
2026-04-15 16:01:53 WARNING  [promptpotter.application.campaign.runner] Optimization interrupted at round 1.



╔════════════════════════════════════════════════════════════════════╗
║  INTERRUPTED — stopped by user                                     ║
╠════════════════════════════════════════════════════════════════════╣
║  Rounds       1              Best         40.0% (round 1)          ║
║  Stop reason  interrupted                                          ║
║  Resume: re-run this cell -- rounds auto-restore                   ║
║  Cycle ID     cycle_0cfa3a4f0136                                   ║
╚════════════════════════════════════════════════════════════════════╝

  Copy-paste pipeline_overrides:
  ────────────────────────────────────────────────────────────
  "pipeline_overrides": {
      "llm_only": {
          "model": 'openai/gpt-oss-120b',
          "temperature": 0.0,
          "max_tokens": 32000,
          "reasoning_effort": 'medium',
      },
  }
  ────────────────────────────────────────────────────────────


KeyboardInterrupt: optimization interrupted after 1 rounds

## Interpretation

`results_potter.json` now sits next to `results_capo.json` and `results_dspy.json` (when those have been run) with an identical top-level schema. The `config.note` field flags that PromptPotter's hyperparameters here are untuned — any head-to-head number below should be read as a floor, not a ceiling, for PromptPotter on BBEH.

Next steps:
- Hyperparameter sweep over `MAX_ROUNDS`, `N_VARIANTS`, `SP_BUDGET_TTEST`.
- Enable sensitivity scan (recon) per task once BBEH-specific `recon_variants.json` is authored.
- Feed the three `results_*.json` files into `docs/research/table-sup-1.md` for the comparison table.